In [1]:
val spark = org.apache.spark.sql.SparkSession.builder()
  .appName("work_sector")
  .master("spark://spark-master:7077") // אם יש לך Master
  .getOrCreate()

org.apache.spark.sql.SparkSession@3414c7c5

### 📥 טעינת הדאטה מ־HDFS

בשלב זה אנו טוענים את הקובץ `datasetN3.txt` ממערכת הקבצים המבוזרת HDFS אל תוך DataFrame ב־Spark.

הגדרות הקריאה כוללות:
- `header = true` – הקובץ מכיל שורת כותרות.
- `inferSchema = true` – Spark ינחש את טיפוסי הנתונים של כל עמודה.
- `sep = ","` – מפריד העמודות הוא פסיק.

לאחר הקריאה, אנחנו מבצעים `cache()` כדי לשמור את הנתונים בזיכרון – פעולה זו משפרת ביצועים בשלבים הבאים של הניתוח.

לבסוף:
- אנו מדפיסים את כמות השורות הכוללת.
- מציגים את הסכמה (schema) של הדאטה.
- מציגים את 5 השורות הראשונות (כולל כל הערכים ללא קיצורים).
  

In [2]:
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .option("sep", ",")
  .csv("hdfs://namenode:8020/data/datasetN3.txt")
  .cache()

println(s"Total rows: ${df.count()}")    
df.printSchema()                         
df.show(5, false)                        

Total rows: 6000
root
 |-- row  number: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- work sector: string (nullable = true)
 |-- salery: integer (nullable = true)
 |-- savings: integer (nullable = true)
 |-- Food expenses (month): integer (nullable = true)
 |-- zip code: integer (nullable = true)
 |-- other expenses (month): integer (nullable = true)
 |-- educaion years: integer (nullable = true)
 |-- economic class: integer (nullable = true)

+-----------+---+-----------+------+-------+---------------------+--------+----------------------+--------------+--------------+
|row  number|age|work sector|salery|savings|Food expenses (month)|zip code|other expenses (month)|educaion years|economic class|
+-----------+---+-----------+------+-------+---------------------+--------+----------------------+--------------+--------------+
|1          |37 |B          |6643  |35501  |1185                 |24602   |1958                  |18            |5             |
|2          |35

df = [row  number: int, age: int ... 8 more fields]


[row  number: int, age: int ... 8 more fields]

### 🧹 ניקוי ראשוני של הנתונים – הסרת אינדקס ותיקון שמות עמודות

בשלב זה אנו מבצעים ניקוי ראשוני ל־DataFrame לאחר טעינת הדאטה המקורית.

הפעולות שבוצעו:

#### ✅ הסרת עמודת אינדקס מיותרת
עמודת `row  number` לא תורמת לניתוח ולכן הוסרה מהדאטה.

#### ✅ שינוי שמות עמודות לשמות תקניים (ללא רווחים ושגיאות כתיב)
לביצוע ניתוח יעיל ונוח יותר, המרה של שמות העמודות בוצעה כך:

| שם מקורי                     | שם חדש             |
|------------------------------|---------------------|
| `work sector`                | `work_sector`       |
| `salery`                     | `salary`            |
| `Food expenses (month)`      | `food_expenses`     |
| `zip code`                   | `zip_code`          |
| `other expenses (month)`     | `other_expenses`    |
| `educaion years`             | `education_years`   |
| `economic class`             | `economic_class`    |

הסיבות לשינוי:
- שמות עם רווחים אינם נוחים לקריאה ועשויים לגרום לשגיאות.
- תיקון שגיאות כתיב משפר עקביות במודל.
- שימוש בקו תחתון `_` הוא סטנדרט נפוץ בעולם ה־Data Engineering.

#### ✅ הצגת הדאטה לאחר הניקוי
לאחר השינויים:
- הדפסת הסכמה המעודכנת (`printSchema`)
- הצגת 5 שורות ראשונות ללא קיצורים (`show(5, false)`)

השלב הזה מבטיח שהדאטה מוכן לעבודה המשמעותית בשלבים הבאים, כמו ניתוח חוסרים, יצירת משתנים חדשים ומידול.

In [3]:
// יבוא פונקציות
import org.apache.spark.sql.functions._

// הסרת העמודה "row  number" + תיקון שמות עמודות
val dfClean = df
  .drop("row  number") // הסרת עמודת אינדקס
  .withColumnRenamed("work sector", "work_sector")
  .withColumnRenamed("salery", "salary")
  .withColumnRenamed("Food expenses (month)", "food_expenses")
  .withColumnRenamed("zip code", "zip_code")
  .withColumnRenamed("other expenses (month)", "other_expenses")
  .withColumnRenamed("educaion years", "education_years")
  .withColumnRenamed("economic class", "economic_class")

// נציג את הסכמה וה-5 שורות הראשונות לאחר השינוי
dfClean.printSchema()
dfClean.show(5, false)

root
 |-- age: integer (nullable = true)
 |-- work_sector: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- savings: integer (nullable = true)
 |-- food_expenses: integer (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- other_expenses: integer (nullable = true)
 |-- education_years: integer (nullable = true)
 |-- economic_class: integer (nullable = true)

+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+
|age|work_sector|salary|savings|food_expenses|zip_code|other_expenses|education_years|economic_class|
+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+
|37 |B          |6643  |35501  |1185         |24602   |1958          |18             |5             |
|35 |E          |7580  |28599  |1960         |21555   |1732          |19             |6             |
|29 |B          |7612  |21559  |1155         |31315   |1294          |18             |7          

dfClean = [age: int, work_sector: string ... 7 more fields]


[age: int, work_sector: string ... 7 more fields]

### הצגת חוסרים אמיתיים של ה-dataset:

מחיקת שורות ריקות לגמרי לקבלת אחוז מדוייק של ערכים חסרים של אותה עמודה

In [4]:
import org.apache.spark.sql.functions._

// תנאי שבודק האם כל התאים בשורה הם null, NaN או ריקים
val allColumnsEmpty = dfClean.columns.map(c =>
  col(c).isNull or isnan(col(c)) or trim(col(c)) === ""
).reduce(_ and _) // שימי לב ל־AND כאן!

// סינון שורות שכל התאים בהן ריקים
val emptyRows = dfClean.filter(allColumnsEmpty)
val emptyCount = emptyRows.count()
println(s"🔎 שורות שרק בהן כל התאים ריקים: $emptyCount")

// מחיקת שורות כאלה בלבד
val dfCleanedStrict = dfClean.filter(!allColumnsEmpty)

// בדיקת כמה שורות נשארו
println(s"✅ מספר שורות לאחר הסרה: ${dfCleanedStrict.count()}")

// הצגת דוגמה
dfCleanedStrict.show(5, false)

🔎 שורות שרק בהן כל התאים ריקים: 14
✅ מספר שורות לאחר הסרה: 5986
+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+
|age|work_sector|salary|savings|food_expenses|zip_code|other_expenses|education_years|economic_class|
+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+
|37 |B          |6643  |35501  |1185         |24602   |1958          |18             |5             |
|35 |E          |7580  |28599  |1960         |21555   |1732          |19             |6             |
|29 |B          |7612  |21559  |1155         |31315   |1294          |18             |7             |
|52 |E          |6958  |44584  |1760         |24157   |1157          |17             |6             |
|38 |D          |2818  |6724   |470          |18148   |1935          |10             |1             |
+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+
only showing top 5

allColumnsEmpty = (((((((((((age IS NULL) OR isnan(age)) OR (trim(age) = )) AND (((work_sector IS NULL) OR isnan(work_sector)) OR (trim(work_sector) = ))) AND (((salary IS NULL) OR isnan(salary)) OR (trim(salary) = ))) AND (((savings IS NULL) OR isnan(savings)) OR (trim(savings) = ))) AND (((food_expenses IS NULL) OR isnan(food_expenses)) OR (trim(food_expenses) = ))) AND (((zip_code IS NULL) OR isnan(zip_code)) OR (trim(zip_code) = ))) AND (((other_expenses IS NULL) OR isnan(other_expenses)) OR (trim(other_expenses) = ))) AND (((education_years IS NULL) OR isnan(education_years)) OR (trim(education_years) = ))) AND (((economic_class IS NULL) OR isnan(economic_class)) OR (trim(economic_class) = )))


emptyRows: org.apach...


(((((((((((age IS NULL) OR isnan(age)) OR (trim(age) = )) AND (((work_sector IS NULL) OR isnan(work_sector)) OR (trim(work_sector) = ))) AND (((salary IS NULL) OR isnan(salary)) OR (trim(salary) = ))) AND (((savings IS NULL) OR isnan(savings)) OR (trim(savings) = ))) AND (((food_expenses IS NULL) OR isnan(food_expenses)) OR (trim(food_expenses) = ))) AND (((zip_code IS NULL) OR isnan(zip_code)) OR (trim(zip_code) = ))) AND (((other_expenses IS NULL) OR isnan(other_expenses)) OR (trim(other_expenses) = ))) AND (((education_years IS NULL) OR isnan(education_years)) OR (trim(education_years) = ))) AND (((economic_class IS NULL) OR isnan(economic_class)) OR (trim(economic_class) = )))

In [5]:
dfCleanedStrict.coalesce(1).write
  .option("header", "true")
  .mode("append")
  .csv("hdfs://namenode:8020/data/df_cleaned_strict_csv_single_file")

In [6]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.Column
val totalRows = dfCleanedStrict.count().toDouble
// הגדרת פונקציה לחוסר (null, "", NaN)
val isNullish: Column => Column = c =>
  c.isNull || trim(c.cast("string")) === "" || isnan(c.cast("double"))

// חישוב עבור כל עמודה
val nullStats = dfCleanedStrict.columns.map { colName =>
  val nullCountCol = sum(when(isNullish(col(colName)), 1).otherwise(0)).alias("null_count")
  val nullDF = dfCleanedStrict.select(nullCountCol)
  val nullCount = nullDF.first().getLong(0)
  val percent = (nullCount / totalRows) * 100

  (colName, nullCount, f"$percent%.2f%%")
}

// הדפסה יפה
println(f"${"Column"}%-25s | ${"Missing"}%-10s | ${"% Missing"}")
println("-" * 50)
nullStats.foreach { case (colName, count, percent) =>
  println(f"$colName%-25s | $count%-10d | $percent")
}

Column                    | Missing    | % Missing
--------------------------------------------------
age                       | 18         | 0.30%
work_sector               | 11         | 0.18%
salary                    | 0          | 0.00%
savings                   | 0          | 0.00%
food_expenses             | 18         | 0.30%
zip_code                  | 18         | 0.30%
other_expenses            | 16         | 0.27%
education_years           | 16         | 0.27%
economic_class            | 46         | 0.77%


totalRows = 5986.0
isNullish = > org.apache.spark.sql.Column = $Lambda$5314/0x0000000101d3f840@205b6fce
nullStats = Array((age,18,0.30%), (work_sector,11,0.18%), (salary,0,0.00%), (savings,0,0.00%), (food_expenses,18,0.30%), (zip_code,18,0.30%), (other_expenses,16,0.27%), (education_years,16,0.27%), (economic_class,46,0.77%))


Array((age,18,0.30%), (work_sector,11,0.18%), (salary,0,0.00%), (savings,0,0.00%), (food_expenses,18,0.30%), (zip_code,18,0.30%), (other_expenses,16,0.27%), (education_years,16,0.27%), (economic_class,46,0.77%))

In [7]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

val salaryWindow = Window.orderBy(col("salary"))

val dfWithExactPercentile = dfCleanedStrict
  .withColumn("exact_percentile", percent_rank().over(salaryWindow) * 100) // אחוזון
  .withColumn("decile", when(col("exact_percentile") <= 10, "0–10%")
    .when(col("exact_percentile") <= 20, "10–20%")
    .when(col("exact_percentile") <= 30, "20–30%")
    .when(col("exact_percentile") <= 40, "30–40%")
    .when(col("exact_percentile") <= 50, "40–50%")
    .when(col("exact_percentile") <= 60, "50–60%")
    .when(col("exact_percentile") <= 70, "60–70%")
    .when(col("exact_percentile") <= 80, "70–80%")
    .when(col("exact_percentile") <= 90, "80–90%")
    .otherwise("90–100%")
  )

dfWithExactPercentile.show(10, false)

+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+-------------------+------+
|age|work_sector|salary|savings|food_expenses|zip_code|other_expenses|education_years|economic_class|exact_percentile   |decile|
+---+-----------+------+-------+-------------+--------+--------------+---------------+--------------+-------------------+------+
|55 |D          |263   |605    |565          |16277   |1603          |16             |0             |0.0                |0–10% |
|38 |D          |635   |1332   |464          |20972   |1224          |11             |0             |0.01670843776106934|0–10% |
|74 |D          |707   |1375   |516          |32025   |1016          |16             |0             |0.03341687552213868|0–10% |
|52 |D          |732   |2309   |431          |33608   |2267          |12             |0             |0.05012531328320802|0–10% |
|68 |D          |747   |1607   |516          |24789   |1870          |15             |0          

salaryWindow = org.apache.spark.sql.expressions.WindowSpec@cd64db1
dfWithExactPercentile = [age: int, work_sector: string ... 9 more fields]


[age: int, work_sector: string ... 9 more fields]

### 💾 שמירת הדאטה עם טווחי עשירונים (deciles) ל־HDFS

בשלב זה שמרנו את טבלת הנתונים dfWithExactPercentile, הכוללת את טווחי העשירון לכל שורת נתונים, אל מערכת הקבצים HDFS.

העמודה decile מתארת את הטווח היחסי של המשתמש לפי השכר (salary), והיא נוספה על בסיס עמודת exact_percentile.

In [8]:
dfWithExactPercentile.write
  .option("header", "true")
  .mode("append") 
  .csv("hdfs://namenode:8020/data/df_with_exact_percentile")

In [9]:
val decileDistribution = dfWithExactPercentile
  .groupBy("decile")
  .count()
  .orderBy("decile")

decileDistribution.show(false)

+-------+-----+
|decile |count|
+-------+-----+
|0–10%  |599  |
|10–20% |599  |
|20–30% |598  |
|30–40% |599  |
|40–50% |598  |
|50–60% |600  |
|60–70% |597  |
|70–80% |599  |
|80–90% |598  |
|90–100%|599  |
+-------+-----+



decileDistribution = [decile: string, count: bigint]


[decile: string, count: bigint]

In [10]:
decileDistribution.write
  .option("header", "true")
  .mode("append") // או "" לפי הצורך
  .csv("hdfs://namenode:8020/outputs/decile_distribution_csv")

In [11]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// סכום לפי עשירון וסקטור
val sectorCounts = dfWithExactPercentile
  .groupBy("decile", "work_sector")
  .agg(count("*").alias("count"))

// סך כל השורות לכל decile – כדי לחשב אחוזים
val totalPerDecile = sectorCounts
  .groupBy("decile")
  .agg(sum("count").alias("total_count"))

// הצטרפות כדי לחשב אחוזים
val sectorWithPercents = sectorCounts
  .join(totalPerDecile, Seq("decile"))
  .withColumn("percent", round(col("count") / col("total_count") * 100, 2))

// דירוג לפי decile
val windowSpec = Window.partitionBy("decile").orderBy(col("percent").desc)

val rankedSectors = sectorWithPercents
  .withColumn("rank", row_number().over(windowSpec))
  .filter(col("rank") <= 2) // רק הסקטור הראשון והשני

// pivot לטבלה סופית: שורה = decile, עמודות = top/second sector + percent
val decileSummaryWithTwoSectors = rankedSectors
  .groupBy("decile")
  .pivot("rank", Seq(1, 2)) // 1 = הכי נפוץ, 2 = השני
  .agg(
    first("work_sector").alias("sector"),
    first("percent").alias("percent")
  )
  .select(
    col("decile"),
    col("1_sector").alias("top_sector"),
    col("1_percent").alias("percent_top_sector"),
    col("2_sector").alias("second_sector"),
    col("2_percent").alias("percent_second_sector")
  )

// הצגה
decileSummaryWithTwoSectors.show(truncate = false)

+-------+----------+------------------+-------------+---------------------+
|decile |top_sector|percent_top_sector|second_sector|percent_second_sector|
+-------+----------+------------------+-------------+---------------------+
|0–10%  |D         |98.0              |C            |1.67                 |
|10–20% |D         |59.93             |C            |39.57                |
|20–30% |C         |78.93             |D            |20.74                |
|30–40% |C         |81.8              |D            |9.85                 |
|40–50% |A         |79.1              |C            |15.89                |
|50–60% |A         |86.0              |B            |13.67                |
|60–70% |B         |65.16             |A            |23.79                |
|70–80% |B         |56.76             |E            |43.07                |
|80–90% |E         |64.05             |B            |35.95                |
|90–100%|E         |75.96             |B            |23.87                |
+-------+---

sectorCounts = [decile: string, work_sector: string ... 1 more field]
totalPerDecile = [decile: string, total_count: bigint]
sectorWithPercents = [decile: string, work_sector: string ... 3 more fields]
windowSpec = org.apache.spark.sql.expressions.WindowSpec@20a58052
rankedSectors = [decile: string, work_sector: string ... 4 more fields]
decileSummaryWithTwoSectors = [decile: string, top_sector: string ... 3 more fields]


[decile: string, top_sector: string ... 3 more fields]

In [12]:
val decileAverages = dfWithExactPercentile
  .groupBy("decile")
  .agg(
    round(avg("salary"), 2).alias("avg_salary"),
    round(avg("education_years"), 2).alias("avg_education_years")
  )
val decileFinalSummary = decileSummaryWithTwoSectors
  .join(decileAverages, Seq("decile"), "left")
  .select(
    col("decile"),
    col("avg_salary"),
    col("avg_education_years"),
    col("top_sector"),
    col("percent_top_sector"),
    col("second_sector"),
    col("percent_second_sector")
  )
decileFinalSummary.show(truncate = false)

+-------+----------+-------------------+----------+------------------+-------------+---------------------+
|decile |avg_salary|avg_education_years|top_sector|percent_top_sector|second_sector|percent_second_sector|
+-------+----------+-------------------+----------+------------------+-------------+---------------------+
|0–10%  |1964.53   |12.6               |D         |98.0              |C            |1.67                 |
|10–20% |2874.13   |13.2               |D         |59.93             |C            |39.57                |
|20–30% |3350.06   |14.08              |C         |78.93             |D            |20.74                |
|30–40% |3787.64   |14.42              |C         |81.8              |D            |9.85                 |
|40–50% |4545.77   |16.11              |A         |79.1              |C            |15.89                |
|50–60% |5229.89   |16.6               |A         |86.0              |B            |13.67                |
|60–70% |6200.65   |17.24            

decileAverages = [decile: string, avg_salary: double ... 1 more field]
decileFinalSummary = [decile: string, avg_salary: double ... 5 more fields]


[decile: string, avg_salary: double ... 5 more fields]

### 📊 סיכום אוכלוסייה לפי עשירונים (`decileFinalSummary`)

הטבלה `decileFinalSummary` כוללת ניתוח מפורט של כלל האוכלוסייה בקובץ, מקובצת לפי טווחי **עשירונים (decile)** שנגזרו על פי השכר החודשי (`salary`).

לכל עשירון מוצגים המדדים הבאים:

| עמודה                  | תיאור                                                                 |
|------------------------|------------------------------------------------------------------------|
| `decile`               | טווח האחוזונים (0–10%, 10–20%, ..., 90–100%) לפי משכורת חודשית.      |
| `count`                | מספר האנשים שנמצאים באותו עשירון.                                     |
| `avg_salary`           | ממוצע השכר באותו עשירון.                                              |
| `avg_education_years`  | ממוצע שנות הלימוד באותו עשירון.                                       |
| `top_sector`           | הסקטור השכיח ביותר באותו עשירון.                                      |
| `percent_top_sector`   | אחוז ההופעה של הסקטור השכיח מתוך כלל הדגימות בעשירון.                |
| `second_sector`        | הסקטור השני בשכיחותו באותו עשירון.                                   |
| `percent_second_sector`| אחוז ההופעה של הסקטור המשני מתוך כלל הדגימות בעשירון.                |

---

🧠 **מטרת הניתוח**: לחשוף תבניות וקשרים בין משתנים מרכזיים:
- משכורת לפי עשירנים ← שנות לימוד
- משכורת לפי עשירונים ← סקטור עבודה
- שכיחות סקטורים ← אופי הפיזור ברמות הכלכליות


In [13]:
decileFinalSummary.write
  .option("header", "true")
  .mode("append")
  .csv("hdfs://namenode:8020/outputs/decile_summary_full")

In [14]:
import org.apache.spark.sql.functions._

val avgEducationBySector = dfWithExactPercentile
  .filter(col("work_sector").isNotNull) // מסנן שורות עם סקטור לא ריק
  .groupBy("work_sector")
  .agg(
    avg("education_years").alias("avg_education_years"),
    stddev("education_years").alias("std_education_years")
  )
  .orderBy("work_sector")

avgEducationBySector.show(false)

+-----------+-------------------+-------------------+
|work_sector|avg_education_years|std_education_years|
+-----------+-------------------+-------------------+
|A          |16.482142857142858 |1.9398672909204335 |
|B          |17.447986577181208 |2.008122709383214  |
|C          |14.416153846153847 |2.0346991404557775 |
|D          |12.503089143865843 |1.9998319690890352 |
|E          |18.44818652849741  |2.0580157581020213 |
+-----------+-------------------+-------------------+



avgEducationBySector = [work_sector: string, avg_education_years: double ... 1 more field]


[work_sector: string, avg_education_years: double ... 1 more field]

In [15]:
avgEducationBySector.write
  .option("header", "true")
  .mode("append") 
  .csv("hdfs://namenode:8020/outputs/avg_education_by_sector")